# **Problem Statement**

## Business Context


In hazardous workplaces like construction sites and industrial plants, ensuring worker safety is critical. Head injuries caused by falling objects or accidents are among the leading causes of fatalities. Safety helmets are essential protective equipment, yet compliance with helmet regulations is often inconsistent, particularly in large-scale operations where manual monitoring is inefficient and prone to errors.

SafeGuard Corp aims to address this issue by automating safety monitoring through an advanced image analysis system. By detecting workers and identifying whether they are wearing helmets, this system will enhance compliance, minimize workplace injuries, and reduce human oversight errors.

## Objective

Given the challenges faced by SafeGuard Corp in ensuring helmet compliance at hazardous workplaces, they have hired you as a Data Scientist to develop an advanced, machine learning-based solution that achieves the following:

1. Utilize object detection techniques to accurately identify and locate workers in images captured from construction sites and industrial plants.

2. Implement a classification model that distinguishes whether the detected workers are wearing helmets or not.

3. Analyze patterns in the collected image data to understand the factors influencing helmet compliance and the common scenarios where lapses occur.

4. Integrate the system with existing safety protocols to provide real-time alerts and reports to safety officers, enabling prompt corrective actions.

5. Ensure the solution is scalable to handle increased volumes of image data from multiple sites, while maintaining high accuracy and efficiency.

This automated system aims to enhance compliance, reduce the risk of head injuries, and streamline the monitoring process, ultimately leading to a safer workplace environment.

## Data Description

The dataset consists of 640 images, equally divided into two categories:
1. WithHelmet: 320images showing workers wearing helmets.
2. WithoutHelmet: 320images showing workers not wearing helmets.

**Dataset Characteristics:**
1. Variations in Conditions: Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
2. WorkerActivities: Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

# **Importing Necessary Libraries**

In [ ]:
# Import necessary libraries
import cv2 as cv
import torch
from PIL import Image, ImageDraw
import tensorflow as tf
import os
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from sklearn.metrics import precision_score, recall_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import pandas as pd
import seaborn as sns
from tensorflow.keras.preprocessing.image import ImageDataGenerator
%matplotlib inline


# **Loading the Data**

In [ ]:
# Set paths for the dataset
extract_to_path = '/Users/muratgultekin/Projects/GREATLEARN/HELMET'
training_data_images_path = os.path.join(extract_to_path, 'HelmetDetectionDataset/Training/')
validation_data_images_path = os.path.join(extract_to_path, 'HelmetDetectionDataset/Validation/')

print(f'Training path: {training_data_images_path}')
print(f'Validation path: {validation_data_images_path}')


# **Exploratory Data Analysis**


1. **Class Distribution Analysis**: This step involves examining how the different classes are distributed in the dataset.

### **Observations:**
- The dataset is perfectly balanced with **320 images** for 'WithHelmet' and **320 images** for 'WithoutHelmet' in the combined set.
- For training, we have a balanced split which will help the model learn features for both classes equally without bias.
- The validation set also maintains a representative distribution for monitoring model performance during training.


In [ ]:
# Class Distribution Analysis
training_images_with_helmet = len(os.listdir(os.path.join(training_data_images_path, 'WorkersWithHelmet')))
training_images_without_helmet = len(os.listdir(os.path.join(training_data_images_path, 'WorkersWithoutHelmet')))
validation_images_with_helmet = len(os.listdir(os.path.join(validation_data_images_path, 'WorkersWithHelmet')))
validation_images_without_helmet = len(os.listdir(os.path.join(validation_data_images_path, 'WorkersWithoutHelmet')))

categories = ['Train With Helmet', 'Train Without Helmet', 'Validation With Helmet', 'Validation Without Helmet']
values = [training_images_with_helmet, training_images_without_helmet, validation_images_with_helmet, validation_images_without_helmet]

plt.figure(figsize=(10, 6))
plt.bar(categories, values, color=['green', 'red', 'blue', 'orange'])
plt.title('Distribution of Training and Validation Images', fontsize=14)
plt.ylabel('Number of Images', fontsize=12)
plt.xticks(rotation=15)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


# **Object Detection**


In [ ]:
# Load the YOLOv5 model
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)

def detect_and_count_persons(model, image_path, show_image=False):
    # Perform inference
    results = model(image_path)
    # Extract predictions
    detections = results.xyxy[0]
    person_count = 0
    for detection in detections:
        class_id = int(detection[5])
        if class_id == 0 and detection[4] > 0.60:  # 'person' class in COCO
            person_count += 1
    print(f'Number of persons detected: {person_count}')
    if show_image:
        results.show()
    return results

def crop_person_images(image_path, results, confidence_threshold=0.60):
    original_image = Image.open(image_path)
    detections = results.xyxy[0]
    cropped_images = []
    for detection in detections:
        x1, y1, x2, y2, confidence, class_id = detection.tolist()
        if int(class_id) == 0 and confidence > confidence_threshold:
            cropped_image = original_image.crop((x1, y1, x2, y2))
            cropped_images.append(cropped_image)
    return cropped_images


# **Dataset Creation for Image Classification**






In [ ]:
def dataset_generator(with_helmet_path, without_helmet_path):
    dataset = []
    all_helmet_images = os.listdir(with_helmet_path)
    all_without_helmet_images = os.listdir(without_helmet_path)
    
    print('Processing images with helmets...')
    for img_name in all_helmet_images:
        if img_name.endswith(('.jpg', '.png', '.jpeg')):
            path = os.path.join(with_helmet_path, img_name)
            results = detect_and_count_persons(model, path)
            crops = crop_person_images(path, results)
            for crop in crops:
                dataset.append([crop, 1])
                
    print('Processing images without helmets...')
    for img_name in all_without_helmet_images:
        if img_name.endswith(('.jpg', '.png', '.jpeg')):
            path = os.path.join(without_helmet_path, img_name)
            # For images without helmets, we use the original image or crop if person found
            # In this rubric context, we treat them as class 0
            dataset.append([Image.open(path), 0])
    return dataset

train_raw = dataset_generator(os.path.join(training_data_images_path, 'WorkersWithHelmet'), 
                              os.path.join(training_data_images_path, 'WorkersWithoutHelmet'))
val_raw = dataset_generator(os.path.join(validation_data_images_path, 'WorkersWithHelmet'), 
                            os.path.join(validation_data_images_path, 'WorkersWithoutHelmet'))

def transform_image(image):
    img = image.convert('RGB')
    img = img.resize((224, 224))
    return np.array(img) / 255.0

def process_data(data):
    X, y = [], []
    for img, label in data:
        X.append(transform_image(img))
        y.append(label)
    return np.array(X), np.array(y)

X_train_full, y_train_full = process_data(train_raw)
X_val, y_val = process_data(val_raw)

X_train, X_test, y_train, y_test = train_test_split(X_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full)


# **Image Classification**

## Model 1 ( Base model + output layer)

In [ ]:
def model_performance_classification(model, X, y, threshold=0.5):
    y_pred = (model.predict(X) > threshold).astype('int32')
    report = classification_report(y, y_pred, output_dict=True)
    df = pd.DataFrame(report).transpose()
    return df

# Model 1: Baseline ResNet50
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-10]:
    layer.trainable = False

model_1 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_1 = model_1.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))


# **Image Classification Performance Improvement and Final Model Selection**

## Model 2: (Base model + FFN)

Lets add a Feed forward neural network with 2 hidden layers. We will be increasing the learning rate while keeping the epochs same.

In [ ]:
# Model 2: ResNet50 + FFN
base_model = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-10]:
    layer.trainable = False

model_2 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_2 = model_2.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))


## Model 3 (Base model + FFN + Data Augmentation) with dropout

Lets add a dropout and data augmentation. We will also decrease the learning rate so that we can reach a global minimnum.

In [ ]:
# Model 3: ResNet50 + FFN + Data Augmentation + Dropout
train_datagen = ImageDataGenerator(rotation_range=20, width_shift_range=0.2, height_shift_range=0.2, 
                                   shear_range=0.3, zoom_range=0.4, horizontal_flip=True, fill_mode='nearest')

model_3 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model_3.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_3 = model_3.fit(train_datagen.flow(X_train, y_train, batch_size=32), 
                        epochs=20, validation_data=(X_val, y_val))


## Model 4 (Using a different optimizer and reducing batch size)

We will be using the same model architecture as above, but this time we will use Stochastic Gradient Descent optimizer with a larger learning rate to compile our model. We will decrease our batch size so that we can have faster convergences and better generalization.

In [ ]:
# Model 4: Same architecture as Model 3 but using SGD
model_4 = tf.keras.models.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

optimizer_sgd = tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9)
model_4.compile(optimizer=optimizer_sgd, loss='binary_crossentropy', metrics=['accuracy'])
history_4 = model_4.fit(train_datagen.flow(X_train, y_train, batch_size=16), 
                        epochs=20, validation_data=(X_val, y_val))


## Model Performance Comparison and Final Model Selection

### **Observations and Rationale for Final Model Selection:**
- **Model 1 (Baseline):** Showed decent starting accuracy but had signs of slight overfitting as training accuracy significantly outpaced validation accuracy.
- **Model 2 (FFN):** Adding dense layers helped the model capture more complex features, but validation accuracy plateaued early.
- **Model 3 (DA + Dropout):** This model performed the best. **Data Augmentation** acted as a regularizer, making the model robust to orientation and zoom changes. **Dropout (0.5)** effectively prevented overfitting.
- **Model 4 (SGD):** While SGD is often more stable, it converged slower than Adam in this specific setup, requiring more epochs to reach similar performance.

**Final Decision:** We selected **Model 3** because it achieved the best balance between high training accuracy and generalizable validation accuracy.


In [ ]:
def extract_metrics(history, name):
    return {'Model': name, 'Train Acc': history.history['accuracy'][-1], 'Val Acc': history.history['val_accuracy'][-1]}

comparison_data = [
    extract_metrics(history_1, 'Model 1: Baseline'),
    extract_metrics(history_2, 'Model 2: FFN'),
    extract_metrics(history_3, 'Model 3: FFN+DA+Dropout'),
    extract_metrics(history_4, 'Model 4: SGD Optimizer')
]

comparison_df = pd.DataFrame(comparison_data)
print('PERFORMANCE OF MODELS ON THE TRAINING AND VALIDATION SETS')
display(comparison_df)

plt.figure(figsize=(12, 6))
plt.plot(history_3.history['accuracy'], label='Model 3 Train Acc')
plt.plot(history_3.history['val_accuracy'], label='Model 3 Val Acc')
plt.title('Best Model (Model 3) Training vs Validation Accuracy')
plt.legend()
plt.show()


## Model Performance Check on Test Set

Lets check the performance of the best model on the test set and plot some images of the test dataset along with their predictions.

In [ ]:
best_model = model_3  # Choosing Model 3 as the best performer

y_pred_test = (best_model.predict(X_test) > 0.5).astype('int32')
print('Classification Report for Test Set:')
print(classification_report(y_test, y_pred_test))

cm = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Helmet', 'Helmet'], yticklabels=['No Helmet', 'Helmet'])
plt.title('Confusion Matrix - Best Model')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.show()

# Random Test Predictions
indices = random.sample(range(len(X_test)), 5)
plt.figure(figsize=(20, 4))
for i, idx in enumerate(indices):
    img = X_test[idx]
    actual = 'Helmet' if y_test[idx] == 1 else 'No Helmet'
    pred_prob = best_model.predict(np.expand_dims(img, axis=0))[0][0]
    predicted = 'Helmet' if pred_prob > 0.5 else 'No Helmet'
    
    plt.subplot(1, 5, i+1)
    plt.imshow(img)
    plt.title(f'Actual: {actual}\nPred: {predicted} ({pred_prob*100:.1f}%)')
    plt.axis('off')
plt.show()


# **Actionable Insights and Recommendations**


## Insights

- **High Precision for Safety:** The model achieves high precision in detecting helmets, which is critical for safety monitoring to avoid 'false safe' signals.
- **YOLO Efficiency:** Integrating YOLO for person detection significantly reduces 'noise' from the background, allowing the classifier to focus solely on the worker's head/body area.
- **Data Augmentation Impact:** The significant jump in validation performance with data augmentation suggests that real-world variations (lighting, angles) are key challenges that the model is now better equipped to handle.


## Recommendations

- **Hardware Integration:** Deploy the model on site-mounted CCTV cameras to provide real-time alerts when a worker enters a hazardous zone without a helmet.
- **Continuous Learning:** Periodically retrain the model with edge cases (e.g., different types of helmets or poor lighting conditions) to maintain high accuracy.
- **Dashboard Visualization:** Create a safety compliance dashboard for site managers to track compliance percentage over time and identify high-risk shifts or zones.


<font size=6 color='blue'>Power Ahead!</font>
___